In [1]:
import os
from dotenv import load_dotenv
import databento as db
import pandas as pd

load_dotenv()
client = db.Historical(os.environ.get("DATABENTO_API_KEY"))

In [ ]:
data = client.timeseries.get_range(
    dataset="XNAS.ITCH",
    symbols="AAPL",
    schema="trades",
    start="2024-01-02",
    end="2025-01-02",
    path="../data/aapl_trades.dbn.zst",
)

store = db.DBNStore.from_file("../data/aapl_trades.dbn.zst")
store.to_parquet("../data/aapl_trades.parquet")

In [ ]:
raw = pd.read_parquet("../data/aapl_trades.parquet")

In [ ]:
# Set the index to the exchange measured timestamp
df = raw.set_index("ts_event")

# Convert the index to the New York timezone
df.index = df.index.tz_convert("America/New_York")

# Filter the DataFrame to only include rows between 9:30 AM and 4:00 PM
df = df.between_time("09:30", "16:00")

In [ ]:
# Resample the DataFrame to 5-minute intervals and take the last price in each interval
prices = df["price"].resample("5min").last()

# Print the first and last timestamps in the resampled DataFrame
print(prices.index[0], prices.index[-1])